# Breakpoints y Criterio de Información Bayesiano (BIC)

> Since `piecewise-regression` calculates the BIC with random initialization, the choice of the **number of breakpoints** must be validated by performing the calculation multiple times (`n_trials`).

---

## Import libraries

In [1]:
import sys
import os
root = os.path.abspath('..')  
sys.path.append(root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from modules import load, plots, analysis
#from modules.archive import processing

In [2]:
print(root)

c:\Users\Mariana\Documents\freshwater_lens


---

## Parameters

In [5]:
# FILE_NAME: Name of the input CSV file containing raw profile measurements (e.g., depth vs conductivity).
FILE_NAME = 'LRS70_D_YSI_R_20250226_processed'
# INPUT_DIR: Folder (relative to project root) where input files are located.
INPUT_DIR = os.path.join(root, 'data', 'rawdy/rawdy_sat51w2p_priority')
# FILE_PATH: Full path to the input file. If you set this explicitly, it takes precedence over INPUT_DIR/FILE_NAME.
FILE_PATH = os.path.join(INPUT_DIR, FILE_NAME + '.csv')

# Model search hyperparameters
# Maximum number of breakpoints (segments - 1). 
MAX_BREAKPOINTS = 10

# Number of random restarts / trials per model size to mitigate local minima.
N_TRIALS = 3

# Output directory for artifacts
OUTPUT_DIR = os.path.join(root, 'data', 'results')

# Column names in the input CSV (edit these if your file headers differ)
VP_NAME = 'Vertical Position m'           # independent variable (x)
SEC_NAME = 'Corrected sp Cond [µS/cm]'     # dependent variable (y)

# Toggle to persist results as JSON to OUTPUT_DIR
SAVE_BIC_RESULTS = True


---

## Load data

In [6]:
# Prefer FILE_PATH if set; otherwise, compose from INPUT_DIR and FILE_NAME
csv_path = FILE_PATH if os.path.isabs(FILE_PATH) or FILE_PATH else os.path.join(INPUT_DIR, FILE_NAME)
print(f"Reading data from: {csv_path}")
df = pd.read_csv(csv_path)

# Extract vectors
x = df[VP_NAME].to_numpy()
y = df[SEC_NAME].to_numpy()

Reading data from: c:\Users\Mariana\Documents\freshwater_lens\data\rawdy/rawdy_sat51w2p_priority\LRS70_D_YSI_R_20250226_processed.csv


---

## Calculate BIC

In [7]:
results = analysis.best_n_breakpoints(x, y, 
                    max_breakpoints=MAX_BREAKPOINTS, 
                    n_trials=N_TRIALS
                    )

Running fit with n_breakpoint = 0 . . 
Running fit with n_breakpoint = 1 . . 
Running fit with n_breakpoint = 2 . . 
Running fit with n_breakpoint = 3 . . 
Running fit with n_breakpoint = 4 . . 
Running fit with n_breakpoint = 5 . . 
Running fit with n_breakpoint = 6 . . 
Running fit with n_breakpoint = 7 . . 
Running fit with n_breakpoint = 8 . . 
Running fit with n_breakpoint = 9 . . 
Running fit with n_breakpoint = 10 . . 

                 Breakpoint Model Comparision Results                 
n_breakpoints            BIC    converged          RSS 
----------------------------------------------------------------------------------------------------
0                 1.8503e+04         True    9.321e+10 
1                 1.8197e+04         True   6.7898e+10 
2                 1.4584e+04         True   1.8577e+09 
3                 1.2412e+04         True   2.1256e+08 
4                 1.1899e+04         True   1.2594e+08 
5                 1.1574e+04         True   9.0046e+07 
6    

---

## Save BIC results

> Calculating different piecewise linear fits is intensive; the information is stored in `JSON` format to avoid repeating this process.

In [8]:
# --- Save results to JSON (optional) ---
if SAVE_BIC_RESULTS:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    results_df = pd.DataFrame(results)
    out_path = os.path.join(OUTPUT_DIR, f"{os.path.splitext(FILE_NAME)[0]}_bic.json")
    results_df.to_json(out_path, indent=2)
    print(f"Saved results to: {out_path}")
else:
    print("Skipping save. Set SAVE_BIC_RESULTS = True to export results as JSON.")

Saved results to: c:\Users\Mariana\Documents\freshwater_lens\data\results\LRS70_D_YSI_R_20250226_processed_bic.json
